In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets.mnist import MNIST
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import numpy as np
import random
import os
import pandas as pd

from matplotlib import pyplot as plt

from GraphRicciCurvature.OllivierRicci import OllivierRicci
import networkx as nx

import sys
sys.path.append("..")

import tools.cnn_adj_matrix as build_cnn_adj
from tools.LeNet5 import LeNet as LeNet
from tools.LeNet5_custom_v2 import LeNet_custom_v2 as LeNet_custom_v2

os.environ['CUDA_VISIBLE_DEVICES'] = '1' 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
seed = 59

# set random seed
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
data_train = MNIST('../data/mnist',
                  train=True,
                  download=True,
                  transform=transforms.Compose([
                      transforms.Resize((32, 32)),
                      transforms.ToTensor()]))

data_test = MNIST('../data/mnist',
                  train=False,
                  download=True,
                  transform=transforms.Compose([
                      transforms.Resize((32, 32)),
                      transforms.ToTensor()]))


In [ ]:
file_path = "edge_v/"
model_path = "models/"

In [ ]:
def get_new_data(l1):
    # selected classes
    train_i1 = torch.tensor([i for i, (_, label) in enumerate(data_train) if label in l1])
    test_i1 = torch.tensor([i for i, (_, label) in enumerate(data_test) if label in l1])
    
    train_index = torch.randperm(len(train_i1))
    valid_dataset = torch.utils.data.Subset(data_train, train_i1[train_index[0:5000]])
    train_dataset = torch.utils.data.Subset(data_train, train_i1[train_index[5000:,]])
    test_dataset = torch.utils.data.Subset(data_test, test_i1)
    
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=1000, num_workers=2)
    valid_loader = DataLoader(valid_dataset, batch_size=1000, num_workers=2)
    
    return train_loader, test_loader, valid_loader, valid_dataset

In [ ]:
def sep_label(dataset, ls):
    sep_dataloader = dict()
    for l in ls:
        index = torch.tensor([i for i, (_, label) in enumerate(dataset) if label == l])
        subset = torch.utils.data.Subset(dataset, index)
        loader = DataLoader(subset, batch_size=5000, num_workers=2)

        sep_dataloader[l] = loader
        
    return sep_dataloader

In [ ]:
selected_classes = [0,1,2,3,4,5,6,7,8,9]
train_loader, test_loader, valid_loader, valid_dataset = get_new_data(selected_classes)

sep_dataloader = sep_label(valid_dataset, selected_classes)


In [ ]:
nodes_num = 9118

model_dims = {
    1: {"name": "input", "dim": {"channel": 1, "out_size": 32}},
    2: {"name": "cnn", "dim": {"channel": 6, "kernel": 5, "stride": 1, "out_size": 28}},
    3: {"name": "pooling", "dim": {"channel": 6, "kernel": 2, "stride": 2, "out_size": 14}},
    4: {"name": "cnn", "dim": {"channel": 16, "kernel": 5, "stride": 1, "out_size": 10}},
    5: {"name": "pooling", "dim": {"channel": 16, "kernel": 2, "stride": 2, "out_size":5}},
    6: {"name": "fc", "dim": {"out_size": 120}},
    7: {"name": "fc", "dim": {"out_size": 84}},
    8: {"name": "fc", "dim": {"out_size": 10}}
}

In [ ]:
model_name= "mnist_relu.pth"
# model_name= "pgdtrain_lenet.pth"
# model_name= "mnist_tanh.pth"
net_H = LeNet_custom_v2(model_dims, None, device)
net_H.load_state_dict(torch.load(model_path + model_name))
net_H = net_H.to(device)

In [ ]:
# Function to perform min-max scaling on weights
def min_max_scale(weights, min_val, max_val):
    scaled_weights = (weights - min_val) / (max_val - min_val)
    return scaled_weights

In [ ]:
adjacent_m, total_e,_ = build_cnn_adj.build_cnn_adj(nodes_num, model_dims, net_H, valid_loader, device)   
        

In [ ]:
def show_results(G, curvature="ricciCurvature"):
    
    neg_e = 0
    edge_set = set()
    most_pos = 0
    nodes = (0,0)
    curs = []

    # Print the first five results
    for n1,n2 in list(G.edges()):
        if (G[n1][n2][curvature] < 0):
            curs.append(G[n1][n2][curvature])
            neg_e += 1
            edge_set.add((n1, n2))
            if (G[n1][n2][curvature] < most_pos):
                most_pos = G[n1][n2][curvature]
                nodes = (n1, n2)
            # print("Ricci curvature of edge (%s,%s) is %f" % (n1 ,n2, G[n1][n2][curvature]))
    print(f'Total number of edges have positive/negative curvature is {neg_e}')
    print(f'The most positive/negative curvature is {most_pos}, between nodes {nodes[0]} and {nodes[1]}, {G[nodes[0]][nodes[1]]["weight"]}.')
    # Plot the histogram of Ricci curvatures
    # plt.subplot(2, 1, 1)
    # ricci_curvtures = nx.get_edge_attributes(G, curvature).values()
    # plt.hist(curs, bins=80)
    # plt.xlabel(f'Ricci curvature: label {label}')
    # plt.title("Histogram of Ricci Curvatures")

    # Plot the histogram of edge weights
    # plt.subplot(2, 1, 2)
    # weights = nx.get_edge_attributes(G, "weight").values()
    # plt.hist(weights,bins=20)
    # plt.xlabel(f'Edge weight for label {label}')
    # plt.title("Histogram of Edge weights")

    # plt.tight_layout()
    # plt.show()
    
    return edge_set, nodes, curs

In [ ]:

# Create network object
G = nx.from_numpy_array(adjacent_m)
print(G)

orf = OllivierRicci(G, alpha=0.5, verbose="TRACE")
# orf.compute_ricci_flow(iterations=100)
orf.compute_ricci_curvature()

G1 = orf.G.copy()
print(G1)


In [ ]:

# edge = sorted(G1.edges(data=True), key=lambda edge: edge[2].get("ricciCurvature", 0), reverse = True)
edge_set, most_nodes, curs = show_results(G1, "ricciCurvature")



In [ ]:
cc = orf.ricci_community_all_possible_clusterings()

In [ ]:
curs.sort(reverse=False)

# Step 3: Calculate the threshold for the top 10%
top_10_percent_count = int(len(curs) * 0.01)

# Step 4: Select the top 10% items
top_10_percent_items = curs[:top_10_percent_count]

print(len(top_10_percent_items))

plt.figure()
n, bins, patches = plt.hist(top_10_percent_items, bins=10, alpha=0.5)
print(n)

plt.xlabel('Ricci curvature')
plt.title("Histogram of Positive Ricci Curvatures")
plt.legend()
plt.show()

In [ ]:
base_edge_s = e_l[0][0]

common_e = base_edge_s & e_l[1][0]
diff_e_l = []

for i in range(2, len(e_l), 1):
    common_e = common_e & e_l[i][0]

for (edge_set, most_nodes, l) in e_l:
    cur_e_s = edge_set
    cur_e_s = cur_e_s - common_e
    diff_e_l.append((l, cur_e_s))

In [ ]:
for (l,sl) in diff_e_l:
    print(len(sl))
    
print(len(common_e))

In [ ]:
# from edge_remove import Edge_Remove

# new_path = model_path

# # remove common edges: # 1
# net_H.load_state_dict(torch.load(model_name))
# edge_r = Edge_Remove(net_H, dims, 28, G1, "ricciCurvature", new_path)

# label_num = len(neg_e_l)

# edge_r.e_remove(common_e, "common")

# # remove all 10 classes edges: #1
# net_H.load_state_dict(torch.load(model_name))
# edge_r = Edge_Remove(net_H, dims, 28, G1, "ricciCurvature", new_path)

# for (edge_set, most_nodes, l) in neg_e_l:
#     edge_r.e_remove(edge_set, "all")

# # remove each class edges: # 10
# for (edge_set, most_nodes, l) in neg_e_l:
#     net_H.load_state_dict(torch.load(model_name))
#     edge_r = Edge_Remove(net_H, dims, 28, G1, "ricciCurvature", new_path)
#     edge_r.e_remove(edge_set, "class" + str(l))

# # remove (each class edges - common edges): # 10
# for (l, edge_set) in diff_e_l:
#     net_H.load_state_dict(torch.load(model_name))
#     edge_r = Edge_Remove(net_H, dims, 28, G1, "ricciCurvature", new_path)
#     edge_r.e_remove(edge_set, "diff" + str(l))

1. XOR curvature

2. 1/edge_weight

3. pick top 1000 hightest edge weight edges

In [ ]:
# remove top 1000 highest edge weights
# 10 classes
# path = "top1000/"
# edges = []

# for (l, G, G1) in G_l:
#     edge = sorted(G1.edges(data=True), key=lambda edge: edge[2].get('weight', 0), reverse = True)[0:1000]
#     edge = list(np.array(edge)[:, 0:2])
#     my_set = {(n1,n2) for (n1,n2) in edge}

#     edges.append((l, my_set))
    

In [ ]:
# compute pair
# 10 labels
# 45 pairs

# from edge_remove import Edge_Remove

# for (edge_set1, most_nodes1, i) in e_l:
#     for (edge_set2, most_nodes2, j) in e_l:
#         if (j != i):
#             common_e = edge_set1 & edge_set2
#             diff_ = edge_set2 - common_e
            
#             net_H.load_state_dict(torch.load(model_path + model_name))
#             edge_r = Edge_Remove(net_H, dims, 28, G1, "ricciCurvature", model_path)
#             edge_r.e_remove(diff_, "pair_" + str(j) + "_" + str(i) + str(j))
            
#             net_H.load_state_dict(torch.load(model_path + model_name))
#             edge_r = Edge_Remove(net_H, dims, 28, G1, "ricciCurvature", model_path)
#             edge_r.e_remove(common_e, "common_" + str(i) + str(j))


1. plot graph

2. pgd: images chaged send to , recalculate edge weight